# Person 1 — Data Layer Pipeline (v2)

### What's new
- **Multi-RPC fallback** — if one provider fails after 5 retries, tries the next automatically
- **Working-RPC memory** — saves which RPC works; re-runs use it first (logged only on change)
- **Single `BLOCK_WINDOW` variable** — change one variable, re-run all cells
- **Concurrent cross-network fetching** — all 3 chains in parallel via ThreadPoolExecutor
- **Raw block CSVs** — `{network}_block{blockNum}_raw.csv` per block
- **BNB error handling** — if Etherscan fails, auto-rechecks all providers and switches

## Step 0 — Environment Setup

In [1]:
import os, sys, json, logging, time
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# SET YOUR BLOCK WINDOW HERE — only variable to change per run
# Recommended: 1, 10, 1000, 10000
# ============================================================
BLOCK_WINDOW = 1

os.environ["BLOCK_WINDOW"] = str(BLOCK_WINDOW)
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)

# Only show INFO for our own modules; suppress noisy third-party logging
logging.basicConfig(level=logging.WARNING,
    format="%(asctime)s [%(levelname)s] %(name)s \u2014 %(message)s", datefmt="%H:%M:%S")
for mod in ["P1_pipeline","src.sampling","src.api_client","src.rpc_registry","src.features"]:
    logging.getLogger(mod).setLevel(logging.INFO)
logger = logging.getLogger("P1_pipeline")

print(f"Project root : {PROJECT_ROOT}")
print(f"Block window : {BLOCK_WINDOW} blocks")

Project root : C:\Users\Pro\Downloads\blockchain_proj\EVM-Cross-Linker
Block window : 1 blocks


## Step 1 — Load Configuration & RPC Registry

In [2]:
from src.config import load_config, APPROX_BLOCK_TIMES, get_window_paths, window_label
from src.rpc_registry import build_all_endpoints, WorkingRPCMemory, order_endpoints_for_chain

cfg = load_config(project_root=PROJECT_ROOT)
cfg.sampling.observation_window_blocks = BLOCK_WINDOW

all_endpoints = build_all_endpoints()
rpc_memory = WorkingRPCMemory(cfg.paths.data_dir)
ordered_endpoints = {
    ch: order_endpoints_for_chain(ch, all_endpoints, rpc_memory)
    for ch in cfg.chains
}

print(f"Block window : {BLOCK_WINDOW} blocks")
print(f"Chains       : {list(cfg.chains.keys())}")
print()
for name, chain in cfg.chains.items():
    providers = [e.name for e in ordered_endpoints.get(name, [])]
    print(f"  {name:12s}  chain_id={chain.chain_id}  RPCs: {', '.join(providers)}")
saved = rpc_memory.get_all()
if saved:
    print(f"\nPreviously working RPCs (tried first): {saved}")

07:59:15 [INFO] src.rpc_registry — Loaded working-RPC memory from C:\Users\Pro\Downloads\blockchain_proj\EVM-Cross-Linker\data\working_rpcs.json


Block window : 1 blocks
Chains       : ['ethereum', 'polygon', 'bnb']

  ethereum      chain_id=1  RPCs: alchemy, nodereal, ankr
  polygon       chain_id=137  RPCs: alchemy, ankr
  bnb           chain_id=56  RPCs: alchemy, chainstack, nodereal, ankr

Previously working RPCs (tried first): {'ethereum': 'etherscan_key1', 'polygon': 'etherscan_key1', 'bnb': 'alchemy'}


## Step 2 — Initialize Client & Connectivity Test

In [3]:
from src.api_client import MultiProviderClient

client = MultiProviderClient(
    api_config=cfg.api,
    chain_endpoints=ordered_endpoints,
    memory=rpc_memory,
)

print("Connectivity test:\n")
chain_heads = {}
for name, chain in cfg.chains.items():
    try:
        latest = client.get_latest_block_number(chain)
        chain_heads[name] = latest
        print(f"  OK   {name:12s}  block #{latest:,}  (via {rpc_memory.get_working_provider(name)})")
    except Exception as e:
        print(f"  FAIL {name:12s}  {e}")
print("\nClient ready.")

Connectivity test:



07:59:16 [INFO] src.rpc_registry — Working RPC for ethereum changed: etherscan_key1 -> alchemy


  OK   ethereum      block #24,739,500  (via alchemy)


07:59:16 [INFO] src.rpc_registry — Working RPC for polygon changed: etherscan_key1 -> alchemy


  OK   polygon       block #84,690,560  (via alchemy)
  OK   bnb           block #88,783,055  (via alchemy)

Client ready.


## Step 3 — Concurrent Block Sampling + EOA Filter

All chains run in parallel. Each saves `{chain}_block{num}_raw.csv`. If a provider fails mid-run, the client auto-falls back.

In [4]:
from src.sampling import run_concurrent_pipeline

t0 = time.time()
print(f"Running pipeline: {BLOCK_WINDOW}-block window, {len(cfg.chains)} chains in parallel\n")

results = run_concurrent_pipeline(client=client, app_config=cfg, window_blocks=BLOCK_WINDOW)

elapsed = time.time() - t0
print(f"\nPipeline complete in {elapsed:.1f}s\n")

all_summaries = []
for chain_name, summary in results.items():
    if "error" in summary:
        print(f"  FAIL  {chain_name}: {summary['error']}")
    else:
        all_summaries.append(summary)
        print(f"  OK  {chain_name:12s}  blocks={summary['blocks_fetched']}  "
              f"active_eoas={summary['active_eoas']}  passive_eoas={summary['passive_eoas']}")

print(f"\n{len(all_summaries)}/{len(cfg.chains)} chains succeeded.")

07:59:17 [INFO] src.sampling — Starting concurrent pipeline: 3 chains, 3 threads, window=1 blocks


Running pipeline: 1-block window, 3 chains in parallel



07:59:18 [INFO] src.sampling — [polygon/1blk] Range [84690560 → 84690560] (1 blocks in window), fetching 1 blocks
07:59:18 [ERROR] src.sampling — [bnb] Pipeline FAILED: Block 88783055 returned null on bnb
Traceback (most recent call last):
  File "C:\Users\Pro\Downloads\blockchain_proj\EVM-Cross-Linker\src\sampling.py", line 574, in run_concurrent_pipeline
    summary = future.result()
  File "c:\Users\Pro\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 449, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\Pro\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\Pro\AppData\Local\Programs\Python\Python313\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\Pro\Downloads\blockchain_proj\EVM-Cross-Linker\src\sampling.py", line 441, in run_chain_pipeline
    latest_pa


Pipeline complete in 59.6s

  FAIL  bnb: Block 88783055 returned null on bnb
  OK  polygon       blocks=1  active_eoas=146  passive_eoas=8
  OK  ethereum      blocks=1  active_eoas=281  passive_eoas=55

2/3 chains succeeded.


### Step 3b — Summary Table (saved to CSV)

In [5]:
wl = window_label(BLOCK_WINDOW)
wpaths = get_window_paths(cfg.paths, BLOCK_WINDOW)

summary_df = pd.DataFrame(all_summaries)
cols = ["chain","window_blocks","blocks_fetched","total_addresses_extracted",
        "pre_eoa_active","pre_eoa_passive","active_eoas","passive_eoas","total_eoas"]
if not summary_df.empty:
    existing = [c for c in cols if c in summary_df.columns]
    print(summary_df[existing].to_string(index=False))
    out_path = cfg.paths.tables_dir / f"person1_sampling_summary_{wl}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path.relative_to(cfg.paths.project_root)}")
else:
    print("No summaries — check errors above.")

   chain  window_blocks  blocks_fetched  total_addresses_extracted  pre_eoa_active  pre_eoa_passive  active_eoas  passive_eoas  total_eoas
 polygon              1               1                        208             149               54          146             8         154
ethereum              1               1                        467             302              138          281            55         336

Saved: outputs\tables\person1_sampling_summary_1blk.csv


## Step 4 — Cross-Chain Transaction Fetching

Errors caught per address. After `RECHECK_THRESHOLD` consecutive failures on a chain, all its endpoints are smoke-tested and `working_rpcs.json` is updated automatically.

In [6]:
# Build master address set
master_addresses = set()
for chain_name in cfg.chains:
    for csv_name in ["active_eoa_addresses.csv", "passive_eoa_addresses.csv"]:
        p = wpaths["interim"] / chain_name / csv_name
        if p.exists():
            df = pd.read_csv(p)
            if "address" in df.columns:
                master_addresses.update(df["address"].dropna().str.lower().tolist())
master_addresses = sorted(master_addresses)
print(f"Master address set: {len(master_addresses)} unique EOAs")

Master address set: 538 unique EOAs


In [7]:
from src.features import fetch_address_chain_activity
from src.rpc_registry import test_endpoint

def recheck_and_update_rpc(chain_name, chain, reason_str):
    """Smoke-tests all endpoints for chain_name and updates working_rpcs.json."""
    print(f"\n  [RPC recheck] {chain_name} — {reason_str[:80]}")
    for ep in ordered_endpoints.get(chain_name, []):
        ok = test_endpoint(ep, timeout=10)
        print(f"    {'OK  ' if ok else 'FAIL'}  {ep.name} ({ep.provider_type})")
        if ok:
            rpc_memory.set_working_provider(chain_name, ep.name)
            ordered_endpoints[chain_name] = order_endpoints_for_chain(
                chain_name, all_endpoints, rpc_memory)
            client.chain_endpoints[chain_name] = ordered_endpoints[chain_name]
            return ep.name
    return None

RECHECK_THRESHOLD = 3
total_jobs = len(master_addresses) * len(cfg.chains)
done = 0
errors = []
consecutive_fails = {n: 0 for n in cfg.chains}
skipped_chains = set()

for address in master_addresses:
    for chain_name, chain in cfg.chains.items():
        if chain_name in skipped_chains:
            continue
        done += 1
        if done % 50 == 0 or done == total_jobs:
            print(f"  [{done}/{total_jobs}] {100*done/total_jobs:.0f}%  "
                  f"last: {address[:12]}... / {chain_name}")
        try:
            fetch_address_chain_activity(
                client=client, app_config=cfg, chain=chain,
                address=address, max_pages_per_endpoint=10,
            )
            consecutive_fails[chain_name] = 0
        except Exception as e:
            err_str = str(e)
            consecutive_fails[chain_name] += 1
            errors.append({"address": address, "chain": chain_name, "error": err_str})
            if consecutive_fails[chain_name] == RECHECK_THRESHOLD:
                new_prov = recheck_and_update_rpc(chain_name, chain, err_str)
                if new_prov:
                    print(f"    Switched {chain_name} to {new_prov}.")
                    consecutive_fails[chain_name] = 0
                else:
                    print(f"    All RPCs dead for {chain_name}. Skipping rest.")
                    skipped_chains.add(chain_name)

print(f"\nDone. OK: {done - len(errors)}/{total_jobs}, Errors: {len(errors)}")
if skipped_chains:
    print(f"Skipped (no working RPC): {skipped_chains}")
if errors:
    err_df = pd.DataFrame(errors)
    print(f"\nErrors by chain:\n{err_df.groupby('chain').size().to_string()}")
    err_path = cfg.paths.tables_dir / f"person1_fetch_errors_{wl}.csv"
    err_df.to_csv(err_path, index=False)
    print(f"Error log: {err_path.relative_to(cfg.paths.project_root)}")

08:00:17 [INFO] src.features — [ethereum] Fetching address activity for 0x0000008e3ed6dd3e870eb169119f1961ba98cfc3
08:00:32 [INFO] src.features — [polygon] Fetching address activity for 0x0000008e3ed6dd3e870eb169119f1961ba98cfc3
08:00:34 [INFO] src.features — [bnb] Fetching address activity for 0x0000008e3ed6dd3e870eb169119f1961ba98cfc3
08:01:12 [WARNING] src.api_client — [bnb/alchemy] alchemy_getAssetTransfers failed: RPC error [alchemy]: {'code': -32602, 'message': "The 'internal' category is only supported for ETH and MATIC."}
08:01:12 [WARNING] src.api_client — [bnb] All Alchemy endpoints failed for address history. Returning empty.
08:01:13 [INFO] src.features — [ethereum] Fetching address activity for 0x000442780bfce7f004ae94657120204851ef8a2e
08:01:14 [INFO] src.features — [polygon] Fetching address activity for 0x000442780bfce7f004ae94657120204851ef8a2e
08:01:16 [INFO] src.features — [bnb] Fetching address activity for 0x000442780bfce7f004ae94657120204851ef8a2e
08:01:18 [WARNIN

  [50/1614] 3%  last: 0x05a2ee196e... / polygon


08:03:20 [INFO] src.features — [bnb] Fetching address activity for 0x05a2ee196eb18c45f7d879ca744b1f4c2dd712e3
08:03:22 [WARNING] src.api_client — [bnb/alchemy] alchemy_getAssetTransfers failed: RPC error [alchemy]: {'code': -32602, 'message': "The 'internal' category is only supported for ETH and MATIC."}
08:03:22 [WARNING] src.api_client — [bnb] All Alchemy endpoints failed for address history. Returning empty.
08:03:22 [INFO] src.features — [ethereum] Fetching address activity for 0x05aa93d7f5dd819511b54582948903024202f272
08:03:23 [INFO] src.features — [polygon] Fetching address activity for 0x05aa93d7f5dd819511b54582948903024202f272
08:03:25 [INFO] src.features — [bnb] Fetching address activity for 0x05aa93d7f5dd819511b54582948903024202f272
08:03:27 [WARNING] src.api_client — [bnb/alchemy] alchemy_getAssetTransfers failed: RPC error [alchemy]: {'code': -32602, 'message': "The 'internal' category is only supported for ETH and MATIC."}
08:03:27 [WARNING] src.api_client — [bnb] All Al

  [100/1614] 6%  last: 0x0d07079639... / ethereum


08:07:45 [WARNING] src.api_client — [ethereum/alchemy] alchemy_getAssetTransfers failed: HTTP 503: {"jsonrpc":"2.0","id":1,"error":{"code":-32000,"message":"Unable to complete request at this time."}}
08:07:45 [WARNING] src.api_client — [ethereum] All Alchemy endpoints failed for address history. Returning empty.


KeyboardInterrupt: 

## Step 5 — Feature Extraction

In [ ]:
from src.features import fetch_and_compute_address_chain_features, feature_records_to_dataframe

window_addresses = set()
for chain_name in cfg.chains:
    for csv_name in ["active_eoa_addresses.csv", "passive_eoa_addresses.csv"]:
        p = wpaths["interim"] / chain_name / csv_name
        if p.exists():
            df = pd.read_csv(p)
            if "address" in df.columns:
                window_addresses.update(df["address"].dropna().str.lower().tolist())
window_addresses = sorted(window_addresses)
total = len(window_addresses) * len(cfg.chains)
print(f"Features: {len(window_addresses)} EOAs x {len(cfg.chains)} chains = {total} jobs\n")

records = []
done = 0
for address in window_addresses:
    for chain_name, chain in cfg.chains.items():
        done += 1
        try:
            rec = fetch_and_compute_address_chain_features(
                client=client, app_config=cfg, chain=chain,
                address=address, max_pages_per_endpoint=10,
            )
            records.append(rec)
        except Exception as e:
            logger.warning("Feature fail %s/%s: %s", address[:12], chain_name, e)
        if done % 50 == 0:
            print(f"  [{done}/{total}] {100*done/total:.0f}%")

ft_df = feature_records_to_dataframe(records)
csv_path = cfg.paths.interim_dir / "feature_rows" / f"address_chain_features_{wl}.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
ft_df.to_csv(csv_path, index=False)
try:
    ft_df.to_parquet(wpaths["processed"] / f"address_chain_features_{wl}.parquet", index=False)
except Exception:
    pass
print(f"\n{len(ft_df)} rows -> {csv_path.relative_to(cfg.paths.project_root)}")

08:09:22 [INFO] src.features — [ethereum] Loading cached address activity for 0x0000008e3ed6dd3e870eb169119f1961ba98cfc3
08:09:22 [INFO] src.features — [polygon] Loading cached address activity for 0x0000008e3ed6dd3e870eb169119f1961ba98cfc3
08:09:22 [INFO] src.features — [bnb] Loading cached address activity for 0x0000008e3ed6dd3e870eb169119f1961ba98cfc3


Features: 538 EOAs x 3 chains = 1614 jobs



08:09:23 [INFO] src.features — [ethereum] Loading cached address activity for 0x000442780bfce7f004ae94657120204851ef8a2e
08:09:23 [INFO] src.features — [polygon] Loading cached address activity for 0x000442780bfce7f004ae94657120204851ef8a2e
08:09:23 [INFO] src.features — [bnb] Loading cached address activity for 0x000442780bfce7f004ae94657120204851ef8a2e
08:09:23 [INFO] src.features — [ethereum] Loading cached address activity for 0x00336a53463f6b6201e726cabf550409140d53a3
08:09:23 [INFO] src.features — [polygon] Loading cached address activity for 0x00336a53463f6b6201e726cabf550409140d53a3
08:09:23 [INFO] src.features — [bnb] Loading cached address activity for 0x00336a53463f6b6201e726cabf550409140d53a3
08:09:23 [INFO] src.features — [ethereum] Loading cached address activity for 0x004bb76d01438b9017a85356fa3734b3d9fcb090
08:09:23 [INFO] src.features — [polygon] Loading cached address activity for 0x004bb76d01438b9017a85356fa3734b3d9fcb090
08:09:23 [INFO] src.features — [bnb] Loading 

  [50/1614] 3%


08:09:24 [INFO] src.features — [polygon] Loading cached address activity for 0x090fc3ead2e5e81d3c0fa2e45636ef003bab9dfb
08:09:24 [INFO] src.features — [bnb] Loading cached address activity for 0x090fc3ead2e5e81d3c0fa2e45636ef003bab9dfb
08:09:24 [INFO] src.features — [ethereum] Loading cached address activity for 0x09277db53892626f4fffef0097ea67a8aa0e93fe
08:09:24 [INFO] src.features — [polygon] Loading cached address activity for 0x09277db53892626f4fffef0097ea67a8aa0e93fe
08:09:24 [INFO] src.features — [bnb] Loading cached address activity for 0x09277db53892626f4fffef0097ea67a8aa0e93fe
08:09:24 [INFO] src.features — [ethereum] Loading cached address activity for 0x0928200e5c298af190bb6afec00a575256adbc5d
08:09:24 [INFO] src.features — [polygon] Loading cached address activity for 0x0928200e5c298af190bb6afec00a575256adbc5d
08:09:24 [INFO] src.features — [bnb] Loading cached address activity for 0x0928200e5c298af190bb6afec00a575256adbc5d
08:09:24 [INFO] src.features — [ethereum] Loading 

### Step 5b — Feature Table Preview

In [ ]:
print(f"Shape: {ft_df.shape}")
if not ft_df.empty:
    agg = ft_df.groupby("chain").agg(
        addresses=("address","nunique"),
        mean_tx=("total_tx_count","mean"),
        median_tx=("total_tx_count","median"),
        has_activity=("has_activity","mean"),
    ).round(2)
    print(agg.to_string())

## Step 6 — Validation & Summaries

In [ ]:
print("OUTPUT FILE CHECK")
print("=" * 60)
expected = []
for cn in cfg.chains:
    for f in ["active_eoa_addresses.csv","passive_eoa_addresses.csv",
              "all_eoa_addresses.csv","pipeline_summary.json"]:
        expected.append(wpaths["interim"] / cn / f)
expected.append(cfg.paths.interim_dir / "feature_rows" / f"address_chain_features_{wl}.csv")
missing = [p for p in expected if not p.exists()]
if not missing:
    print(f"  All {len(expected)} expected files present.")
else:
    for p in missing:
        print(f"  MISSING: {p.relative_to(cfg.paths.project_root)}")
raw_csvs = list((wpaths["raw"] / "block_samples").rglob("*_block*_raw.csv"))
print(f"\nRaw block CSVs: {len(raw_csvs)} files")
if raw_csvs:
    print(f"  Example: {raw_csvs[0].name}")

In [ ]:
ap_rows = []
for cn in cfg.chains:
    act_p = wpaths["interim"] / cn / "active_eoa_addresses.csv"
    pas_p = wpaths["interim"] / cn / "passive_eoa_addresses.csv"
    n_a = len(pd.read_csv(act_p)) if act_p.exists() else 0
    n_p = len(pd.read_csv(pas_p)) if pas_p.exists() else 0
    ap_rows.append({"window_blocks": BLOCK_WINDOW, "chain": cn,
                    "active_eoas": n_a, "passive_eoas": n_p,
                    "total": n_a + n_p,
                    "active_pct": round(100 * n_a / max(n_a + n_p, 1), 1)})
ap_df = pd.DataFrame(ap_rows)
print("ACTIVE / PASSIVE SUMMARY")
print(ap_df.to_string(index=False))
ap_df.to_csv(cfg.paths.tables_dir / f"person1_active_passive_{wl}.csv", index=False)

In [ ]:
print("WORKING RPCs AFTER THIS RUN")
for ch, prov in rpc_memory.get_all().items():
    print(f"  {ch:12s} -> {prov}")

## Step 7 — Visualizations

In [ ]:
import matplotlib.pyplot as plt
if not ap_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    x = range(len(ap_df))
    ax.bar(x, ap_df["active_eoas"], label="Active", color="#2196F3", alpha=0.85)
    ax.bar(x, ap_df["passive_eoas"], bottom=ap_df["active_eoas"],
           label="Passive", color="#FF9800", alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(ap_df["chain"].tolist(), rotation=30)
    ax.set_title(f"Active vs Passive EOAs — {BLOCK_WINDOW}-block window")
    ax.set_ylabel("EOA count")
    ax.legend()
    plt.tight_layout()
    fig.savefig(cfg.paths.figures_dir / f"person1_active_passive_{wl}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
if not ft_df.empty:
    chains_in = ft_df["chain"].unique()
    fig, axes = plt.subplots(1, len(chains_in), figsize=(5 * len(chains_in), 4), sharey=True)
    if len(chains_in) == 1:
        axes = [axes]
    for ax, cn in zip(axes, chains_in):
        vals = ft_df[ft_df["chain"] == cn]["total_tx_count"]
        ax.hist(vals, bins=30, color="#4CAF50", alpha=0.7, edgecolor="white")
        ax.set_title(f"{cn} ({BLOCK_WINDOW:,}blk)")
        ax.set_xlabel("Total Tx Count")
    fig.suptitle(f"Tx Count Distribution ({BLOCK_WINDOW:,}-block window)", fontsize=13, y=1.02)
    plt.tight_layout()
    fig.savefig(cfg.paths.figures_dir / f"person1_tx_dist_{wl}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

## Done — Deliverables

| File | Location |
|------|----------|
| `active_eoa_addresses.csv` | `data/interim/{N}blk/{chain}/` |
| `passive_eoa_addresses.csv` | `data/interim/{N}blk/{chain}/` |
| `{chain}_block{num}_raw.csv` | `data/raw/{N}blk/block_samples/{chain}/` |
| `address_chain_features_{N}blk.csv` | `data/interim/feature_rows/` |
| `person1_sampling_summary_{N}blk.csv` | `outputs/tables/` |
| `person1_active_passive_{N}blk.csv` | `outputs/tables/` |
| `person1_fetch_errors_{N}blk.csv` | `outputs/tables/` (only if errors) |
| `working_rpcs.json` | `data/` |

To run a different window: change `BLOCK_WINDOW` in Step 0, re-run all cells.

In [ ]:
print(f"Block window : {BLOCK_WINDOW}")
print(f"Chains       : {list(cfg.chains.keys())}")
print(f"Master EOAs  : {len(master_addresses)}")
print(f"Features     : {ft_df.shape[0]} rows x {ft_df.shape[1]} cols")
print(f"Working RPCs : {rpc_memory.get_all()}")
client.close()
print("Client closed. Done.")